# 3.0 - Aritmética de punto flotante

## Parte 1: Preguntas conceptuales

Responda las siguientes preguntas en forma clara y concisa.

### 1. Representación binaria vs representación decimal.

Explique por qué la expresión 0.1 + 0.2 == 0.3 da el resultado False en Python. No responda simplemente “por errores de redondeo”; explique los detalles específicos sobre cómo 0.1 es representado en notación binaria comparada con su representación decimal.

Los números flotantes son representados por la suma de fracciones del tipo $\frac 1{2^n}$, por lo que los únicos numeros que pueden reprensentarse de manera exacta deben tener como denominador una potencia de dos, lo cual no ocurre para $\frac1{10}, \frac15$ y $\frac3{10}$. Esto significa que los tres numeros estan representados de manera erronea y sucede que la suma de los primeros dos errores no es lo mismo que el tercero.

In [10]:
0.1 + 0.2 == 0.3

False

### 2. Falta de asociatividad.
En matemáticas, la suma de números reales es asociativa, es decir, si , tenemos que
. Explique por qué esta propiedad no siempre se cumple en aritmética de punto flotante. Dé un ejemplo cualitativo de un escenario en donde el orden de la adición podría cambiar el resultado significativamente.

Podemos hacer una opreción del tipo $a+b-c = x$ donde $x$ puede reprensentarse sin problema pero $a+b>inf$ desborda los límites de bits que pueden guardarse. El orden de la operación previene este escenario si se cambia a, por ejemplo $a-c+b$.

In [17]:
import numpy as np
eps_64 = np.finfo(np.float64).eps
real_max_64 = (2-eps_64)*2**1023
a = real_max_64
b = real_max_64
c = real_max_64 -5

In [18]:
a + b - c

inf

In [19]:
a - c + b

1.7976931348623157e+308

### 3. Cancelación catastrófica.

Defina “Cancelación catastrófica” con sus propias palabras. ¿Por qué este tipo de error en específico es considerado más peligroso que el error de redondeo estándar? Use TecGPT para encontrar ejemplos típicos.

Este es un tipo de error que ocurre al restar dos números casi iguales. Los dígitos significativos de ambos números se pierden en la resta y lo único que queda son los bits más pequeños asociados a los errores de redondeo. es más peligroso que el error de redondeo estándar porque en el error estándar al menos el error de redondeo está ajustado a la escala del número (su exponente), en este caso, puede que al restar dos números muy grandes el resultado sea ruido que se hace pasar por dígitos significativos.

**TECGPT**:

La cancelación catastrófica ocurre cuando restas (o sumas con signos opuestos) dos números muy parecidos en punto flotante y el resultado “pierde” la mayoría de sus cifras significativas.
En otras palabras: la resta borra los dígitos buenos y lo que queda está dominado por el ruido de redondeo.

In [1]:
import math

x = 1e-8
naive = 1 - math.cos(x)
stable = 2 * math.sin(x/2)**2  # identidad trigonométrica equivalente

print("naive :", naive)
print("stable:", stable)
print("rel error:", abs(naive - stable) / stable)

naive : 0.0
stable: 5.0000000000000005e-17
rel error: 1.0


In [2]:
import math

a, b, c = 1.0, 1e8, 1.0

# fórmula clásica
disc = math.sqrt(b*b - 4*a*c)
x1_naive = (-b + disc) / (2*a)
x2_naive = (-b - disc) / (2*a)

# forma estable: usar -b - disc para una raíz y luego x2 = c/(a*x1)
x1_stable = (-b - disc) / (2*a)
x2_stable = c / (a * x1_stable)

print("naive x1 :", x1_naive)
print("stable x2:", x2_stable)

naive x1 : -7.450580596923828e-09
stable x2: -1e-08


In [3]:
import math

x = 1e16
naive = math.sqrt(x + 1) - math.sqrt(x)

# racionalización (multiplicar por el conjugado)
stable = 1 / (math.sqrt(x + 1) + math.sqrt(x))

print("naive :", naive)
print("stable:", stable)
print("rel error:", abs(naive - stable) / stable)

naive : 0.0
stable: 5e-09
rel error: 1.0


### 4. Diferenciación numérica.
Cuando se calcula una derivada numéricamente usando la fórmula de diferencias finitas
$$f'(x) ≈ \frac{f(x+h)-f(x)}{h}$$
en principio se desea que $h$ sea tan pequeño como sea posible para minimizar el error de truncamiento. Sin embargo, si hacemos $h$ demasiado pequeño, por ejemplo, $10^{-15}$, el error en realidad crece drásticamente. ¿Por qué?

Al evaluar la fórmula de diferencias finitas existen **dos fuentes de error que compiten**:

1. **Error de truncamiento** (Taylor): al expandir $f(x+h)$ en serie de Taylor, el término dominante que se omite es $\frac{h}{2}f''(x)$, por lo que este error es $O(h)$ y *decrece* conforme $h \to 0$.

2. **Error de redondeo** (cancelación catastrófica): cuando $h$ es muy pequeño, $f(x+h)$ y $f(x)$ son casi idénticos. La diferencia $f(x+h)-f(x)$ pierde casi todos sus dígitos significativos. Si el error absoluto de evaluar $f$ es $\sim \varepsilon_{\text{maq}}\,|f(x)|$ (con $\varepsilon_{\text{maq}} \approx 2.2 \times 10^{-16}$ para float64), el error en la diferencia es $\sim 2\,\varepsilon_{\text{maq}}\,|f(x)|$, y al dividir entre $h$ crece como $\dfrac{2\,\varepsilon_{\text{maq}}\,|f(x)|}{h}$.

El error total es la suma de ambas contribuciones:
$$E(h) \approx \underbrace{\frac{h}{2}|f''(x)|}_{\text{truncamiento}} + \underbrace{\frac{2\,\varepsilon_{\text{maq}}\,|f(x)|}{h}}_{\text{redondeo}}$$

Minimizando en $h$ se obtiene el **paso óptimo**:
$$h^* \approx \sqrt{\varepsilon_{\text{maq}}} \approx 10^{-8}$$

Con $h = 10^{-15}$ estamos muy por debajo del óptimo: la cancelación catastrófica domina y el error crece como $1/h$, destruyendo por completo la aproximación.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

f  = np.sin
df = np.cos          # derivada exacta de sin
x0 = 1.0

hs     = np.logspace(-16, 0, 300)
errors = np.abs((f(x0 + hs) - f(x0)) / hs - df(x0))

eps   = np.finfo(float).eps
h_opt = np.sqrt(eps)

plt.figure(figsize=(7, 4))
plt.loglog(hs, errors, lw=1.5, label='error numerico')
plt.axvline(h_opt, color='r', ls='--',
            label=f'h_opt = sqrt(eps) ~ {h_opt:.0e}')
plt.xlabel('h'); plt.ylabel('error absoluto')
plt.title('Error en diferencias finitas adelante para f=sin')
plt.legend(); plt.grid(True, which='both', alpha=0.3)
plt.tight_layout(); plt.show()

print(f'Error con h=1e-8  : {abs((f(x0+1e-8)  - f(x0))/1e-8  - df(x0)):.2e}')
print(f'Error con h=1e-15 : {abs((f(x0+1e-15) - f(x0))/1e-15 - df(x0)):.2e}')

### 5. Raíz cuadrada inversa.
Utilizando TecGPT busque información sobre el algoritmo para calcular el inverso de la raíz cuadrada de un número (Fast Inverse Square Root). Basándose en dicha información, escriba un breve texto donde incluya el algoritmo, una descripción de los pasos que aprovechan las características de números de punto flotante, y ejemplos de implementación en videojuegos "retro".

## Fast Inverse Square Root

El **Fast Inverse Square Root** (FISR) es un algoritmo que calcula $y \approx x^{-1/2}$ aprovechando directamente la estructura interna del formato IEEE 754.

### ¿Por qué es útil?
En motores de videojuegos se necesita normalizar vectores con mucha frecuencia (iluminación, física). Normalizar requiere $\|v\|^{-1} = (v_x^2+v_y^2+v_z^2)^{-1/2}$. En hardware de los años 90 la división y la raíz cuadrada eran decenas de veces más lentas que las operaciones enteras.

### El truco de los bits IEEE 754
Un `float32` almacena: $x = (1 + m/2^{23}) \cdot 2^{e-127}$, con $m$ la mantisa entera y $e$ el exponente entero. Tomando $\log_2$:
$$\log_2 x \approx \frac{e \cdot 2^{23} + m}{2^{23}} - 127 = \frac{I_x}{2^{23}} - 127$$
donde $I_x$ es la **reinterpretación entera** de los bits del float. En otras palabras, el entero $I_x$ codifica aproximadamente $\log_2 x$.

Para $y = x^{-1/2}$:
$$\log_2 y = -\tfrac{1}{2}\log_2 x \implies I_y \approx \tfrac{3}{2} \cdot 127 \cdot 2^{23} - \tfrac{I_x}{2} = \texttt{0x5f3759df} - (I_x \gg 1)$$

La constante mágica `0x5f3759df` es una leve corrección sobre $\frac{3}{2} \cdot 127 \cdot 2^{23}$ que minimiza el error máximo de la aproximación inicial.

### Algoritmo completo (C original de Quake III Arena, 1999)
```c
float Q_rsqrt(float number) {
    long i;
    float x2, y;
    const float threehalfs = 1.5F;

    x2 = number * 0.5F;
    y  = number;
    i  = *(long *) &y;           // reinterpretar bits del float como entero
    i  = 0x5f3759df - (i >> 1);  // primera aproximacion en dominio entero
    y  = *(float *) &i;          // volver a float
    y  = y * (threehalfs - (x2 * y * y));  // una iteracion de Newton-Raphson
    return y;
}
```

### Pasos que explotan punto flotante

| Paso | Qué hace | Por qué funciona |
|------|----------|-----------------|
| `*(long*)&y` | copia bits sin conversión | reinterpreta la representación IEEE 754 como entero |
| `0x5f3759df - (i>>1)` | opera en enteros | los bits aproximan $\log_2$, restar mitad divide el exponente entre 2 |
| `*(float*)&i` | vuelve a float | el resultado ya es buena aproximación en IEEE 754 |
| Newton-Raphson | refina | una sola iteración lleva el error relativo a $\lesssim 0.2\%$ |

### Uso en videojuegos retro
- **Quake III Arena (id Software, 1999)**: encontrado en el código fuente publicado del motor, popularmente atribuido a John Carmack aunque investigaciones apuntan a Terje Mathisen y Gary Tarolli. Se usaba para calcular iluminación por vértice y normales de superficie en tiempo real.
- **GLQuake / Half-Life (Valve, 1998)**: derivados del motor Quake emplearon variantes del mismo truco para física y renderizado en hardware sin FPU rápida dedicada.
- En hardware moderno la instrucción `RSQRTSS` (SSE) hace algo análogo en silicio, pero el FISR sigue siendo referencia clásica de ingeniería creativa con punto flotante.

In [ ]:
import struct, math

def fast_inv_sqrt(number):
    x2 = number * 0.5
    packed = struct.pack('f', number)
    i = struct.unpack('I', packed)[0]
    i = 0x5f3759df - (i >> 1)          # magia de bits
    y = struct.unpack('f', struct.pack('I', i))[0]
    y = y * (1.5 - (x2 * y * y))       # Newton-Raphson
    return y

for v in [1.0, 4.0, 9.0, 100.0, 0.25]:
    approx  = fast_inv_sqrt(v)
    exact   = 1.0 / math.sqrt(v)
    rel_err = abs(approx - exact) / exact
    print(f'x={v:7.2f}  FISR={approx:.8f}  exacto={exact:.8f}  error={rel_err:.2e}')

## Parte 2: Ejercicios computacionales

Use el archivo 1 - Floating point arithmetic.py para implementar sus soluciones.

### 1. La serie armónica
La serie armónica (finita) está definida por $\sum_{n=1}^N \frac1n$. Dado que la precisión de punto flotante es finita, el orden en que se suman los términos de la serie es importante.

1. Implemente la función ***harmonic_sum_forward(n)***. Esta función debe calcular la suma de la serie armónica de $1$ a $N$ (en ese orden).
2. Implemente la función ***harmonic_sum_backward(n)***. Esta función debe calcular la suma de la serie armónica de $N$ a $1$ (es decir, en orden inverso).
3. Ejecute ambas funciones para $N=1000000$ (1 millón).
4. Análisis: ¿Cuál método es más preciso y por qué? El valor de la serie del punto anterior, con $32$ cifras significativas, es $14.392726722865723631381127493189$.

### 2. Cálculo estable de la varianza
La fórmula estándar para calcular la varianza de un conjunto de datos $X = \{x_1, x_2, \ldots, x_N\}$ es
$$var(X) = \frac1N \sum_{i=1}^N (x_i- \bar x)^2$$
Con frecuencia, en los libros de estadística se ofrece una fórmula alternativa “de un paso” que es más barata computacionalmente,
$$var(X) = \frac1N \sum_{i=1}^N x_i^2- (\bar x)^2$$
Sin embargo, la segunda fórmula es numéricamente inestable cuando la varianza es pequeña comparada con la magnitud de los datos (por ejemplo, si $X = \{1000000.1, 1000000.2, \ldots\}$
).

1. Implemente la función ***variance_naive(data)*** usando la fórmula inestable de un paso.
2. Implemente la función ***variance_stable(data)*** usando el algoritmo de dos pasos (calcular la media aritmética primero y luego iterar la suma de diferencias cuadradas) o el algoritmo de Welford (busque información sobre este último).
3. Pruebe ambas funciones con el conjunto de datos dado en el archivo adjunto.

### 3. La fórmula cuadrática revisitada
En clase usted aprendió que la fórmula cuadrática estándar falla cuando $b^2 ≈ 4ac$.

1. Implemente la función ***solve_quadratic(a, b, c)*** que devuelve una tupla con las raíces ***(x1, x2)***.
2. Su función implementada debe detectar cuál raíz causa cancelación catastrófica (basada en el signo de $b$) y use la fórmula cuadrática alterna para esa raíz en específico.
3. Pruebe su función con $a=1 , \ b=10^8, \ c=1$.